# Code Review Agent (Notebook)

Minimal agentic demo that reviews a pull request and posts a comment using:
- An **OpenAI-compliant** inference endpoint secured with API Key
- MCP Endpoint secured with API Key. **2 MCP tools**: `get_pull_request_files`, `create_issue_comment`

Set env vars (or override in the config cell): `OPENAI_API_BASE`, `OPENAI_API_KEY`, optionally `MCP_PROXY_URL`, `MCP_PROXY_BEARER_TOKEN`, `OPENAI_MODEL`.

## 1. Imports and config

In [ ]:
import json
import requests
from openai import OpenAI

# Config (from env; override below for notebook)
MCP_URL = "https://agents.ai.nutanix.com/enterpriseai/mcp/user-45-mcp-connector"
MCP_BEARER_TOKEN = "<your-mcp-api-key>"

OPENAI_MODEL = "oss-demo-1"
OPENAI_BASE = "https://agents.ai.nutanix.com/enterpriseai/v1/chat/completions"
OPENAI_KEY = "<inference-api-key>"



print(f"MCP_URL: {MCP_URL}")
print(f"OPENAI_BASE: {OPENAI_BASE or '(not set)'}")
print(f"OPENAI_MODEL: {OPENAI_MODEL}")

MCP_URL: https://agents.ai.nutanix.com/enterpriseai/mcp/user-45-mcp-connector
OPENAI_BASE: https://agents.ai.nutanix.com/enterpriseai/v1/chat/completions
OPENAI_MODEL: oss-demo-1


## 2. MCP client

In [85]:
# Session ID from server; capture from initialize (and any) response, send on all subsequent requests.
MCP_SESSION_ID = [None]


def _mcp_headers():
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if MCP_BEARER_TOKEN:
        headers["Authorization"] = f"Bearer {MCP_BEARER_TOKEN}"
    if MCP_SESSION_ID[0]:
        headers["Mcp-Session-Id"] = MCP_SESSION_ID[0]
    return headers


def _capture_session_id(resp) -> None:
    """Extract MCP session ID from response headers or body and store for subsequent requests."""
    sid = resp.headers.get("Mcp-Session-Id") or resp.headers.get("mcp-session-id")
    if not sid and (resp.text or "").strip():
        try:
            data = _parse_mcp_response_body(resp.text)
            result = data.get("result") if isinstance(data, dict) else data
            if isinstance(result, dict):
                sid = result.get("sessionId") or result.get("session_id")
        except Exception:
            pass
    if sid:
        MCP_SESSION_ID[0] = sid


def _parse_mcp_response_body(text: str) -> dict:
    """Parse MCP response: plain JSON or SSE (event: message, data: {...})."""
    if not text or not text.strip():
        return {}
    text = text.strip()
    # Plain JSON
    if text.startswith("{"):
        return json.loads(text)
    # SSE: lines like "event: message" and "data: {...}"
    if "data:" in text:
        data_parts = []
        for line in text.split("\n"):
            line = line.strip()
            if line.startswith("data:"):
                data_parts.append(line[5:].strip())
        if data_parts:
            json_str = "\n".join(data_parts)
            if json_str:
                return json.loads(json_str)
    return json.loads(text)


def mcp_request(method: str, params: dict | None = None) -> dict:
    payload = {"jsonrpc": "2.0", "id": 1, "method": method}
    if params is not None:
        payload["params"] = params
    headers = _mcp_headers()
    resp = requests.post(MCP_URL, json=payload, headers=headers, timeout=30)
    _capture_session_id(resp)
    resp.raise_for_status()
    try:
        data = _parse_mcp_response_body(resp.text or "")
    except json.JSONDecodeError:
        raise RuntimeError(f"MCP response is not JSON. Status: {resp.status_code}. Body: {(resp.text or '')[:300]!r}")
    if "error" in data:
        raise RuntimeError(data["error"].get("message", data["error"]))
    return data.get("result", {})


def mcp_initialize():
    return mcp_request("initialize", {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "code-review-agent", "version": "1.0"},
    })


def mcp_list_tools() -> list:
    result = mcp_request("tools/list")
    all_tools = result.get("tools", [])
    return all_tools

def mcp_call_tool(name: str, arguments: dict) -> str:
    result = mcp_request("tools/call", {"name": name, "arguments": arguments})
    for part in result.get("content", []):
        if part.get("type") == "text":
            return part.get("text", "")
    return ""

## 3. MCP → OpenAI tools and system prompt

In [86]:
def mcp_tools_to_openai(mcp_tools: list) -> list:
    openai_tools = []
    for t in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t.get("description", ""),
                "parameters": t.get("inputSchema", {"type": "object", "properties": {}}),
            },
        })
    return openai_tools


SYSTEM_PROMPT = """You are a code review agent. You can get pull request files. If you have the pull request files, you can create a review comment on a pull request.
Your task: Review the code for the given PR and then post a single, concise review comment summarizing your findings (what looks good, any suggestions or concerns). Use the tools available to you in order. First get the PR files, go through them, then write the review comment. After posting the comment, reply with a short confirmation to the user."""

## 4. Agent loop

In [87]:
def run_agent(owner: str, repo: str, pull_number: int) -> str:
    if not OPENAI_BASE:
        raise ValueError("Set OPENAI_API_BASE to your OpenAI-compliant inference endpoint URL")

    # prepare OpenAI client for Inference
    base_url = OPENAI_BASE.rsplit("/chat/completions", 1)[0] if "/chat/completions" in OPENAI_BASE else OPENAI_BASE.rstrip("/")
    client = OpenAI(base_url=base_url, api_key=OPENAI_KEY)

    # initialize MCP connection
    mcp_initialize()
    mcp_tool_list = mcp_list_tools()
    openai_tools = mcp_tools_to_openai(mcp_tool_list)

    user_message = (
        f"Please review the code for this pull request and add a comment with your review.\n"
        f"Repository: {owner}/{repo}\n"
        f"Pull request number: {pull_number}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    final_content = ""

    # when to stop the agent loop
    max_turns = 10

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto",
        )
        choice = response.choices[0]
        msg = choice.message
        finish = choice.finish_reason

        # Print inference response
        out = {"finish_reason": finish, "content": msg.content or None}
        if getattr(msg, "tool_calls", None):
            out["tool_calls"] = [{"id": tc.id, "function": tc.function.name, "arguments": (tc.function.arguments or "")[:200]} for tc in msg.tool_calls]

        if msg.content:
            final_content = (msg.content or "").strip()

        if finish == "stop" and not getattr(msg, "tool_calls", None):
            return final_content or "Review flow finished."

        tool_calls = getattr(msg, "tool_calls", None) or []
        if not tool_calls:
            return final_content or "Review flow finished."

        assistant_msg = {
            "role": "assistant",
            "content": msg.content or None,
            "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in tool_calls
            ],
        }
        messages.append(assistant_msg)

        for tc in tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments) if isinstance(tc.function.arguments, str) else tc.function.arguments
            except json.JSONDecodeError:
                args = {}
            try:
                result = mcp_call_tool(name, args)
            except Exception as e:
                result = json.dumps({"error": str(e)})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return final_content or "Max turns reached."

## 5. Run the agent

In [88]:
owner = "octocat"      # repo owner
repo = "hello-world"   # repo name
pull_number = 123       # PR number

result = run_agent(owner, repo, pull_number)
print(result)

I was able to fetch the changed files and see that the only modification is an added “Hola Mundo!” line in the README. The change is harmless and improves the documentation for Spanish‑speaking users.

I attempted to post a review comment, but the request was rejected with a 403 error (“Resource not accessible by personal access token”). It appears my token does not have permission to create PR reviews in this repository.

If you grant the appropriate write permissions, I can repost the review comment. Otherwise, you can copy the comment below and add it manually:

> Nice addition to the README! Adding the Spanish “Hola Mundo!” line improves accessibility for Spanish‑speaking users. No code changes are affected, so this looks good to merge.
